# 君正股份董秘问答 LoRA 微调（Colab + 免费 T4）

目标：用小模型（默认 Qwen2.5-1.5B-Instruct）在「投资者提问 -> 董秘作答」数据上做 LoRA/QLoRA 微调，
让模型学会君正股份董秘的披露口径。

步骤：安装依赖 -> 上传数据 -> 加载模型并挂 LoRA -> SFT 训练 -> 保存 adapter -> 推理测试。

> 说明：官方目前没有名为 `Qwen3.5-2B` 的模型；这里默认用稳定存在的 `Qwen2.5-1.5B-Instruct`，
> 你也可以把 `MODEL_ID` 换成 `Qwen/Qwen3-1.7B` 等其它小模型。

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import os

# 可免费 T4 运行的小模型
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_PATH = "/content/qa_dataset_clean.jsonl"
OUTPUT_DIR = "/content/junzheng_lora"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
MAX_LEN = 1024
EPOCHS = 2
BATCH = 2
GRAD_ACC = 4
LR = 2e-4

### 1. 上传数据

上传本地仓库里的 `data/qa_dataset_clean.jsonl`（抽检后的干净数据，485 条）。

In [ ]:
from google.colab import files

print("请上传 data/qa_dataset_clean.jsonl")
uploaded = files.upload()
for name in uploaded:
    DATA_PATH = "/content/" + name
print("DATA_PATH =", DATA_PATH)

In [ ]:
import json
from datasets import Dataset

SYSTEM = (
    "你是A股上市公司北京君正(君正股份,300223)的董事会秘书。"
    "请用合规、客观、克制的董秘口吻回答投资者问题，只依据公司财报信息作答。"
)

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]

def to_qa(r):
    return {
        "instruction": SYSTEM,
        "input": "投资者提问：" + r["question"],
        "output": r["answer"],
    }

ds = Dataset.from_list([to_qa(r) for r in rows]).train_test_split(test_size=0.15, seed=42)
print(ds)

### 2. 加载模型 + 挂 LoRA（QLoRA 4bit）

T4 是 Turing 架构，不支持 bf16，因此统一用 fp16。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### 3. SFT 训练

In [ ]:
import torch
from transformers import Trainer, TrainingArguments

def tokenize_fn(examples):
    texts = []
    for ins, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        msgs = [
            {"role": "system", "content": ins},
            {"role": "user", "content": inp},
            {"role": "assistant", "content": out},
        ]
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    enc = tokenizer(texts, truncation=True, padding="max_length", max_length=MAX_LEN)
    labels = []
    for ids, mask in zip(enc["input_ids"], enc["attention_mask"]):
        labels.append([-100 if m == 0 else t for t, m in zip(ids, mask)])
    enc["labels"] = labels
    return enc

tok_ds = ds.map(tokenize_fn, batched=True, remove_columns=ds["train"].column_names)
tok_ds.set_format("torch")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_ds["train"],
    eval_dataset=tok_ds["test"],
)
trainer.train()


### 4. 保存 LoRA adapter

In [ ]:
adapter_dir = os.path.join(OUTPUT_DIR, "adapter")
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("LoRA adapter saved to", adapter_dir)

# 可选：把 adapter 合并进基座，得到完整模型（体积更大）
# merged = model.merge_and_unload()
# merged.save_pretrained(os.path.join(OUTPUT_DIR, "merged"))

### 5. 推理测试

In [ ]:
def ask(question):
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user", "content": "投资者提问：" + question}],
        tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for q in ["公司2025年的营业收入是多少？", "2026年上半年净利润为什么大幅增长？"]:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 60)

## 备注

- T4(16GB) 跑 1.5B 模型 + QLoRA 足够；2 个 epoch、batch=2、grad_acc=4 大约几分钟到十几分钟。
- 想换更大模型可把 `MODEL_ID` 改成 `Qwen/Qwen3-1.7B`；若用 A100/L4，可把 fp16 换成 bf16 并把 compute_dtype 改为 `torch.bfloat16`。
- 训练数据越少，越容易过拟合；29 条主要做“口径示范”，正式使用建议用 `generate_qa.py` 扩量到数百条。